In [743]:
import math 
import random
import hashlib

# ICC132 -Actividad integradora: implementación de un esquema asimétrico de firma y verificación

## Codigo Fuente

In [ ]:
#Revisa si el numero es primo o no, regresa true o false
def ifPrimo(num):
    if num <= 1:
        return False
    for i in range(2, int(math.sqrt(num)) + 1):
        if num % i == 0:
            return False  
    return True

#regresa un numero primo al azar entre los dos valores dados
def numPrimo(minVal, maxVal):
    primo = random.randint(minVal, maxVal)
    valid = ifPrimo(primo)
    while valid == False:
        primo = random.randint(minVal, maxVal)
        valid = ifPrimo(primo)
    return primo

#Genera p q primos y distintos
def pqDist(min, max): 
    p = numPrimo(min, max)
    q = numPrimo(min, max)
    while p == q:
        q = numPrimo(min, max)
    return p,q


#genera llaves utilizando el algoritmo de RSA
def llaveRSA(min,max):
    #Se generan p,q (primos y distintos)
    p,q = pqDist(min, max)

    #producto de primo
    n = p * q 

    #calcular Euler's Totient (n)=(p-1)x(q-1)
    euler = (p - 1) * (q - 1)

    #*escoger encryption exponent (e) que cumpla con 
    # 1 < e < euler
    #gcd(e, euler) = 1, e debe ser co primo con euler.
    e = random.randrange(2, euler)
    while math.gcd(e, euler) != 1:
        e = random.randrange(2, euler)

    #calcular el multiplicativo inverso, la funcion pow nativa de python
    # calcula utiliza el Extended Euclidean Algorithm al pasarle el -1
    d = pow(e, -1, euler) 
    
    #llaves 
    public = (e,n)
    private = (n,d)

    return public, private

#Transforma el mensaje a un numero entero, despues de aplicar un SHA256 y transformar el hex 
def hashing(texto):
    texto = str(texto)
    hash = hashlib.sha256(texto.encode('utf-8'))
    hex = hash.hexdigest()    
    numEntero = int(hex, 16)
    return numEntero

#Crea la firma del texto que recibe utilizando la llave privada
def signRSA(texto,llavePriv):
    n , d = llavePriv
    entero = hashing(texto)
    cutEntero = entero % n #se recorta ya que el resultado del hash es muy grande y esto que siempre es menor a n 
    signed = pow(cutEntero, d, n)
    return signed

#Verfica si el texto es el correcto con la llave publica
def verif(texto,sign,llavePublic):
    try: 
        e , n = llavePublic
        entero = hashing(texto)
        cutEntero = entero % n
        signedText = pow(sign, e , n)
        if (cutEntero == signedText):
            return True
        else:
            return False
    except Exception as e:
        print("Error de formato")



In [745]:
min= 1000
max= 9999
print("Verificacion Normal")
public, private = llaveRSA(min,max)
print("La llave publica es", public)
print("La llave privada es ", private)

texto="hola"
signed = signRSA(texto,private)
print("El texto inicial es", texto, "\nSu firma es ", signed)

print("Verificando con la llave", public)

print(verif(texto,signed,public))

Verificacion Normal
La llave publica es (11729167, 17621459)
La llave privada es  (17621459, 8249503)
El texto inicial es hola 
Su firma es  10638551
Verificando con la llave (11729167, 17621459)
True


## Documento breve de diseño

### Esquema elegido

Se escogio el algoritmo de RSA para el algoritmo asincrono por su base en aritmética modular y la factorización de números primos, lo cual simplifica la implementacion en python, sin hacer uso de librerias externas. En este caso solo se utilizaron las librerias de math,random y hashlib.

### Descripción general del flujo de firma y verificación

**Flujo de Firma:** Se toma el documento (texto original) y se pasa por una funcion de hashing para obtener un numero entero que lo represente, ya que el RSA no funciona sobre strings y se requiere transformarlo a un entero con la funcion de hashing. Esta  implementada de manera que transforma un texto a SHA 256, el cual esta representado como un mensaje hexadecimal, se transforma un valor entero. Por limitaciones del RSA y al no usar valores primo iniciales muy grandes se requiere reducir el tamaño del entero por que no puede ser mayor a n **(n = p*q)**. Posteriormente, utiliza su Llave Privada obtenida anteriormente para encriptar este número 

> signed = pow(cutEntero, d, n)


**Flujo de verificacion:** Para verificar que el texto original recibido es el aprobado se calcula su hash de forma independiente. Simultaneamente, toma la firma matematica y utiliza la Llave Publica dada para desencriptar. Finalmente, compara ambos resultado. Si son idénticos, regresa 'True' verificando que el documento no fue alterado y que proviene del autor legitimo.

### Explicación de las llaves

Para obtener las llaves se realizo los pasos para obtener llaves de RSA explicados en Geeksforgeeks. 

Para empezer se requiere que p y q sean primos distintos, y se obtiene el producto primo obtienendo para darle valor a n (n = p * q), posteriermente se calcula el valor totiente de euler expresado por euler = (p - 1) * (q - 1) para escoger un exponente de encriptacion el cual debe ser mayor a uno y menor que el valor de euler y  ademas que ambos coincidan en tener un maximo comun divisor de 1. Con el objetivo que el exponente sea coprimo del valor de euler.

Por otro lado, el exponente de desencriptación ($d$) se obtiene calculando el inverso multiplicativo modular del exponente público ($e$) obtenido anteriormente, de tal  que se cumpla la congruencia matemática $e \times d \equiv 1 \pmod{\phi(n)}$. Para resolver esto computacionalmente, se aprovechó el Algoritmo de Euclides Extendido, el cual viene implementado y optimizado de forma nativa en Python mediante la función    
> pow(e, -1, euler).

Con el exponente de encriptacion y el exponente desencriptación, se pueden ahora conseguir las llaves publicas y las llaves privadas. Las que vienen definidas como 

>public = (e,n)

>private = (n,d)

Fuente de la explicacion del algoritmo RSA utilizado:

https://www.geeksforgeeks.org/computer-networks/rsa-algorithm-cryptography/

### Representación del mensaje

El texto original dado el cual se asume que es un string, se tiene que transformar a un valor entero numerico para aplicarle las relaciones matematicas que depende el algoritmo de RSA. En mi implementacion se realize el proceso empezando con el texto ingresado se codifica primero a bytes (UTF-8) y se procesa mediante el algoritmo SHA-256. Esta operación devuelve una cadena en formato hexadecimal. Finalmente, mediante la instrucción int(hex, 16), esta cadena se convierte en un número entero gigante de base 10. Esta representación numérica es la que permite que el mensaje interactúe con la función pow() para realizar las ecuaciones criptográficas.

### Validaciones implementadas

La primera validacion es en asegurarse que solo se trabajen sobre numeros primos y distintos para que la generacion de llaves funcione correctamente por lo que se sigue corriendo el ciclo en caso de regrese 2 numeros primos iguales. 

Otra verificacion es traducir todo el texto dado a un string, ya que las transformaciones empiezan asumiendo que el valor es un string, la funcion de hashing automaticamente traduce todo valor a un string y consequentemente a un hash numerico. 

Ademas en la ultima funcion de verificacion se calcula de nuevo la firma y se compara con la firma dada. Asegurando que el mensaje original no tuvo cambio ademas de que la firma sea la real. Y en caso de que algun dato dado no sea valido con la verificacion regresa un error.

### Limitaciones y observaciones técnicas


**Llaves factorizables:** Por propositos educativos y de tiempo de ejecucion el generador de primos regresa numeros primos pequeño . Esto genera un módulo n pequeño (de 5 o 6 cifras) que cualquier analizador moderno podría factorizar en segundos para deducir la llave secreta.

**Recorte del Hash:** Al generar un módulo n pequeño y un hash SHA-256 gigantesco, se rompió la regla de RSA que dicta que el mensaje debe ser menor a n. Para evitar un desbordamiento, se implementó la instrucción cutEntero = entero % n. Este recorte fuerza al hash a caber en la llave educativa, lo cual preserva la mecánica de las fórmulas, pero debilita drásticamente la singularidad de la huella digital, aumentando la probabilidad de colisiones matemáticas.

## Bitácora de desarrollo



| Etapa | Problema Encontrado | Decisión Tomada | Ajuste Realizado | Explicación de la Decisión |
| :--- | :--- | :--- | :--- | :--- |
| **Generacion de Primos** | Los números aleatorios $p$ y $q$ resultaban ser iguales en ocasiones. | Forzar que los factores primos sean estrictamente distintos. | Se añadió un ciclo `while p == q` en la función `pqDist`. | El algoritmo de RSA requiere que p y q sean valores distintos y el algoritmo no se corria correctamente en los raros casos que al azar regresara dos numero iguales. |
| **Procesamiento de Hash** | El SHA-256 generaba numeros demasiado grandes para el modulo $n$ educativo. | Reducir el tamaño de los hashes para que el mensaje sea menor a $n$. | Se aplico `cutEntero = entero % n` antes de firmar y verificar. | Inicialmente trate un corte de bits simple, pero esto rompía el algoritmo y causo aun mas problemas. Al buscar en linea soluciones el modulo ajustaba el valor sin romper la logica de RSA. Ademas se asegura que el hash siempre va a ser menor que n. |
| **Funcion de Verificacion** | El programa regresaba (`AttributeError`) al recibir tipos de datos inesperados (no-strings). | Implementar bloqueos para que el codigo de verificacion detecte cuando surga un error. | Se envolvió la lógica en un bloque `try...except` y se validó el formato de entrada. | Un sistema de seguridad no debe detenerse abruptamente ante errores de entrada, sino manejarlos y denegar la firma limpiamente. |
| **Testing** | Al ingresar valores que no eran strings, el algoritmo de hashing no corria correctamente ya que esperaba strings declaradas. | Convertir todos los datos a un string antes de empezar |  Se implemento `texto = str(texto)` para transformar los datos que se le paso a un string  | Traducir los datos a un string fue la opcion para hacer que el algoritmo corra fuera de error de usuario y aun firma datos vacios|

## Verificaciones

#### Verficacion con un mensaje vacio

In [746]:
print("Verificacion con mensaje vacio")
public, private = llaveRSA(min,max)
print("La llave publica es ",public)
print("La llave privada es ",private)

texto=""
signed = signRSA(texto,private)
print("El texto inicial es", texto, "\nSu firma es ", signed)
print("Verificando con la llave", public)

print(verif(texto,signed,public))

Verificacion con mensaje vacio
La llave publica es  (12816439, 50551507)
La llave privada es  (50551507, 48331079)
El texto inicial es  
Su firma es  9885871
Verificando con la llave (12816439, 50551507)
True


In [747]:
print("Verificacion con mensaje vacio")
public, private = llaveRSA(min,max)
print("La llave publica es ",public)
print("La llave privada es ",private)

texto=None
signed = signRSA(texto,private)
print("El texto inicial es", texto, "\nSu firma es ", signed)
print("Verificando con la llave", public)

print(verif(texto,signed,public))

Verificacion con mensaje vacio
La llave publica es  (22144541, 39198619)
La llave privada es  (39198619, 55397)
El texto inicial es None 
Su firma es  14477188
Verificando con la llave (22144541, 39198619)
True


#### El mensaje cambia despues de generar la firma

In [748]:
public, private = llaveRSA(min,max)
print("La llave publica es", public)
print("La llave privada es ", private)

texto="hola "
signed = signRSA(texto,private)
print("El texto inicial es", texto, "\nSu firma es ", signed)

texto = "hola" #sin espacio
print("El segundo texto es", texto, "\nVerificando con la llave", public)

print(verif(texto,signed,public))

La llave publica es (8062871, 21329351)
La llave privada es  (21329351, 19331471)
El texto inicial es hola  
Su firma es  12532958
El segundo texto es hola 
Verificando con la llave (8062871, 21329351)
False


#### Entradas no validas

In [749]:
print("Entradas no validas")
public, private = llaveRSA(min,max)
print("La llave publica es", public)
print("La llave privada es ", private)

texto="hola"
signed = signRSA(texto,private)
print("El texto inicial es", texto, "\nSu firma es ", signed)
print("Verificando con la llave", public)

print(verif(texto,"4824",public))

Entradas no validas
La llave publica es (9039639, 30512281)
La llave privada es  (30512281, 3585367)
El texto inicial es hola 
Su firma es  30180055
Verificando con la llave (9039639, 30512281)
Error de formato
None


In [750]:
print("Entradas no validas")
public, private = llaveRSA(min,max)
print("La llave publica es", public)
print("La llave privada es ", private)

texto="hola"
signed = signRSA(texto,private)
print("El texto inicial es", texto, "\nSu firma es ", signed)
print("Verificando con la llave", public)

print(verif(texto,signed,None))

Entradas no validas
La llave publica es (78878395, 90935287)
La llave privada es  (90935287, 42136811)
El texto inicial es hola 
Su firma es  1856069
Verificando con la llave (78878395, 90935287)
Error de formato
None


#### Verficacion si la llave publica cambia

In [751]:
print("Verificacion cambiando la llave publica")
public, private = llaveRSA(min,max)
print("La llave publica es", public)
print("La llave privada es ", private)

texto="hola"
signed = signRSA(texto,private)
print("El texto inicial es", texto, "\nSu firma es ", signed)

e , n = public
e = e + 1
n = n + 1
public = e , n 

print("La llave publica modificada es", public)


print("Verificando con la llave", public)
print(verif(texto,signed,public))

Verificacion cambiando la llave publica
La llave publica es (1037053, 7538291)
La llave privada es  (7538291, 3500677)
El texto inicial es hola 
Su firma es  1448002
La llave publica modificada es (1037054, 7538292)
Verificando con la llave (1037054, 7538292)
False


#### Verficacion si la firma no coincide

In [752]:
public, private = llaveRSA(min,max)
print("La llave publica es", public)
print("La llave privada es ", private)

texto="hola"
signed = signRSA(texto,private)
print("El texto inicial es", texto, "\nSu firma es ", signed)

signed = signed + 1 
print("Firma modificada", signed)
print("Utilizando la llave: ", signed)

print(verif(texto,signed,public))

La llave publica es (13103343, 38456701)
La llave privada es  (38456701, 190527)
El texto inicial es hola 
Su firma es  6968742
Firma modificada 6968743
Utilizando la llave:  6968743
False
